# Atividade: Classificação de Grãos

## Dataset
Para esta tarefa, utilizaremos uma versão adaptada do Dry Bean Dataset, disponível no arquivo `beans_train.csv`. O conjunto é composto por amostras de diferentes tipos de feijões secos, descritas por atributos morfológicos extraídos a partir de imagens, como área, perímetro, compacidade, excentricidade e fatores de forma. Essas características capturam propriedades geométricas relevantes dos grãos, permitindo distinguir classes com base em padrões estruturais no espaço de atributos.

As amostras estão organizadas em sete classes distintas de feijões: Seker, Barbunya, Bombay, Cali, Dermason, Horoz e Sira.

## O que será avaliado
Serão considerados os seguintes aspectos:

* **Implementação:** Coerência entre o problema proposto e o método de classificação adotado, bem como a correta aplicação do algoritmo.
* **Estabilidade:** Controle do comportamento do processo de treinamento, assegurando convergência consistente e reduzindo variações indesejadas ao longo da otimização.
* **Regularização:** Capacidade de mitigar o sobreajuste por meio de estratégias adequadas ao modelo e aos dados.
* **Análise de Desempenho:** Interpretação dos resultados com base em métricas apropriadas de avaliação.

## Entrega
A atividade deve ser entregue obrigatoriamente em formato de **Jupyter Notebook (.ipynb)**. O arquivo deve estar devidamente organizado e comentado, justificando as decisões tomadas para garantir a estabilidade e a capacidade de generalização do modelo.


# Desenvolvimento da Atividade

## Investigação preliminar dos dados

Buscamos compreender de forma basica a caracteristica dos dados do dataset beans_train.csv.
Para isso vamos ler o dataset num dataframe pandas e levantar algumas estatisticas.

In [1]:
import torch
import pandas as pd

df = pd.read_csv('beans_train.csv')

# Seleciona o dispositivo com prioridade: CUDA > MPS > CPU
device = torch.device(
    'cuda' if torch.cuda.is_available() # GPU
    else 'mps' if torch.backends.mps.is_available() # MPS (Mac Silicon)
    else 'cpu' # CPU
)
print(f"Usando o dispositivo: {device}")

Usando o dispositivo: mps


### Visualizando as primeiras 10 linhas

In [2]:
df.head(10)

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,38465.705594,737.090445,273.111968,179.201525,1.523333,0.753998,38999.014066,220.991553,0.813252,0.988670,0.888416,0.809359,0.007117,0.001879,0.655710,0.996874,SIRA
1,70247.179155,1019.740591,376.308440,238.192942,1.582115,0.773556,71562.831696,299.139162,0.707757,0.985750,0.850248,0.794947,0.005351,0.001324,0.632118,0.998320,BARBUNYA
2,74212.949558,1049.756998,414.858346,229.655950,1.810139,0.834413,75004.806409,306.719019,0.776048,0.987679,0.844313,0.739884,0.005597,0.001039,0.548725,0.991096,CALI
3,31233.029736,671.584675,257.751304,154.448259,1.670969,0.800540,31682.423625,198.813755,0.718182,0.988299,0.870828,0.773574,0.008257,0.001822,0.598230,0.996528,DERMASON
4,48189.750788,826.752018,313.328739,196.497443,1.604009,0.781591,48840.530648,247.963648,0.796755,0.988498,0.887039,0.788613,0.006525,0.001556,0.622442,0.995508,SIRA
5,57356.907479,958.191465,400.523353,182.796156,2.186816,0.889473,57860.551234,269.985000,0.569907,0.988100,0.782668,0.673195,0.007018,0.000892,0.453699,0.992911,HOROZ
6,32171.814046,659.848322,239.925576,171.207725,1.408751,0.704095,32773.991337,203.133528,0.732065,0.986681,0.926887,0.842839,0.007454,0.002320,0.708291,0.997717,DERMASON
7,54115.074552,855.886477,286.734712,242.350112,1.180368,0.531732,54686.273502,262.825642,0.788499,0.990122,0.933415,0.919372,0.005270,0.002320,0.845115,0.997845,SEKER
8,77259.272204,1096.973650,385.738417,256.126058,1.508870,0.748752,78685.329186,314.300183,0.773244,0.983759,0.809262,0.812457,0.004993,0.001348,0.658929,0.995432,BARBUNYA
9,36349.710092,689.671173,242.197615,190.452403,1.270238,0.616005,36493.668120,214.491109,0.776832,0.990745,0.952517,0.886411,0.006695,0.002553,0.786668,0.998273,SEKER


### Calculando a distribuição das featureas: médias, desvio padrão, percentis, máximos e mínimos

In [3]:
df.describe()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4
count,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000,10888.000000
mean,52984.826554,854.735445,319.900759,202.160613,1.582773,0.750914,53707.155435,252.908484,0.749728,0.987122,0.873351,0.799940,0.006567,0.001718,0.643693,0.995069
std,29295.457791,214.444640,85.691172,44.964220,0.245892,0.091630,29749.674675,59.182734,0.048963,0.004668,0.059242,0.061495,0.001130,0.000596,0.098626,0.004390
min,20345.408013,524.528134,183.312292,122.327171,1.026275,0.218787,20684.208989,161.412105,0.555519,0.943587,0.571799,0.645318,0.002777,0.000568,0.416878,0.950032
25%,36287.942022,702.783268,253.116922,175.613608,1.433148,0.716372,36637.411296,214.876017,0.718193,0.985636,0.832720,0.762659,0.005905,0.001155,0.581743,0.993741
50%,44534.203579,794.101721,296.618790,192.380899,1.549896,0.764116,45050.873268,238.145830,0.759660,0.988270,0.883286,0.801594,0.006648,0.001700,0.642496,0.996399
75%,61301.065303,976.352085,376.380761,216.986950,1.706149,0.810119,62219.957215,279.298556,0.786905,0.989989,0.916905,0.834040,0.007280,0.002173,0.695535,0.997896
max,254495.210567,1987.238396,738.618173,460.043477,2.389710,0.908263,263278.394353,569.109998,0.866303,0.994666,0.990888,0.987507,0.010451,0.003664,0.975513,0.999727


### Encontrando as features com maior e menor média e desvio padrão

In [4]:
def explore_dataset(s):
  '''
  Retorna o índice e o valor mínimo e máximo de um série
  '''
  return s.idxmin(), s.min(), s.idxmax(), s.max()

numeric_df = df.drop(columns=['Class'])
# médias de cada atributo numérico
means = numeric_df.mean()

# mínimo e máximo entre as médias
min_means_attr, min_means_value, max_means_attr, max_means_value = explore_dataset(means)

# desvios padrão de cada atributo numérico
deviations = numeric_df.std()
min_deviations_attr, min_deviations_value, max_deviations_attr, max_deviations_value = explore_dataset(deviations)

print(f"Menor média........: {min_means_attr} = {min_means_value:.4f}")
print(f"Menor desvio padrão: {min_deviations_attr} = {min_deviations_value:.6f}")
print( "Atributo com a menor média é igual ao atributo com o menor desvio padrão" if min_means_attr == min_deviations_attr 
      else "Atributo com a menor média e atributo com o menor desvio padrão são diferentes")
print()
print(f"Maior média........: {max_means_attr} = {max_means_value:.2f}")
print(f"Maior desvio padrão: {max_deviations_attr} = {max_deviations_value:.2f}")
print( "Atributo com a maior média é igual ao atributo com o maior desvio padrão" if max_means_attr == max_deviations_attr 
      else "Atributo com a maior média e atributo com o maior desvio padrão são diferentes")

Menor média........: ShapeFactor2 = 0.0017
Menor desvio padrão: ShapeFactor2 = 0.000596
Atributo com a menor média é igual ao atributo com o menor desvio padrão

Maior média........: ConvexArea = 53707.16
Maior desvio padrão: ConvexArea = 29749.67
Atributo com a maior média é igual ao atributo com o maior desvio padrão


In [5]:
df.info()
print(f"O dataset possui {df.shape[0]} amostras e {df.shape[1]} atributos: shape {df.shape}")

<class 'pandas.DataFrame'>
RangeIndex: 10888 entries, 0 to 10887
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Area             10888 non-null  float64
 1   Perimeter        10888 non-null  float64
 2   MajorAxisLength  10888 non-null  float64
 3   MinorAxisLength  10888 non-null  float64
 4   AspectRation     10888 non-null  float64
 5   Eccentricity     10888 non-null  float64
 6   ConvexArea       10888 non-null  float64
 7   EquivDiameter    10888 non-null  float64
 8   Extent           10888 non-null  float64
 9   Solidity         10888 non-null  float64
 10  roundness        10888 non-null  float64
 11  Compactness      10888 non-null  float64
 12  ShapeFactor1     10888 non-null  float64
 13  ShapeFactor2     10888 non-null  float64
 14  ShapeFactor3     10888 non-null  float64
 15  ShapeFactor4     10888 non-null  float64
 16  Class            10888 non-null  str    
dtypes: float64(16), str(1)


### Observações importantes

A partir da exploração inicial do dataset, destacam-se os seguintes pontos:

1. O conjunto de dados possui 10888 amostras, com 16 atributos (features) e 1 variável de classe, totalizando 17 colunas (shape: 10888 × 17).
2. Não foram identificados valores nulos em nenhuma das colunas, não sendo necessária, portanto, uma etapa de limpeza preliminar.
3. Todos os 16 atributos são numéricos, do tipo *float64*, eliminando a necessidade de pré-processamento para codificação de variáveis não numéricas.
4. Observa-se uma diferença significativa de escala entre os atributos; a feature com menor média e desvio padrão (ShapeFactor2) apresenta $\mu=0.0017$ e $\sigma=0.000596$, enquanto a feature com maior média e desvio padrão (ConvexArea) apresenta $\mu=53707.16$ e $\sigma=29749.67$.

Diante desse cenário, evidencia-se a necessidade de normalização dos dados de entrada, a fim de mitigar o viés decorrente das diferenças de escala entre os atributos.

*IMPORTANTE:* Por questões metodológicas, os dados devem ser particionados em conjuntos de treino, validação e teste. Os parâmetros de normalização (média $\mu$ e desvio padrão $\sigma$) serão estimados exclusivamente a partir do conjunto de treino e, em seguida, aplicados de forma consistente aos conjuntos de validação e teste. Esse procedimento busca evitar o vazamento de informação (*data leakage*) e garantir a comparabilidade entre os conjuntos durante o processo de treinamento e avaliação dos modelos.

### Particionando os dados

Vamos particionar os dados nos conjuntos de treino (80%), validação (10%) e teste (10%), mantendo a estratificação das espécies para preservar a representatividade dos dados.

In [6]:
from sklearn.model_selection import train_test_split

SEED = 42

# separa features e classe
X = df.drop(columns=['Class'])
y = df['Class']

# 1) treino (80%) e temporário (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

# 2) validação (10%) e teste (10%)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,   # metade de 20% = 10%
    stratify=y_temp,
    random_state=SEED
)

print(f"len(X_train)={len(X_train)}, len(X_validation)={len(X_validation)}, len(X_test)={len(X_test)}")

len(X_train)=8710, len(X_validation)=1089, len(X_test)=1089


#### Conferindo a estratificação das espécies

A estratégia de amostragem estratificada preservou a distribuição das classes entre os conjuntos de treino, validação e teste, apresentando variações desprezíveis decorrentes apenas do arredondamento no particionamento das amostras, como podemos observar a seguir:

In [11]:
dist = pd.DataFrame({
    "Original": y.value_counts(),
    "Original (%)": y.value_counts(normalize=True),
    "Treino": y_train.value_counts(),
    "Treino (%)": y_train.value_counts(normalize=True),
    "Validação": y_validation.value_counts(),
    "Validação (%)": y_validation.value_counts(normalize=True),
    "Teste": y_test.value_counts(),
    "Teste (%)": y_test.value_counts(normalize=True),
})

dist

,Original,Original (%),Treino,Treino (%),Validação,Validação (%),Teste,Teste (%)
Class,,,,,,,,
DERMASON,2849,0.261664,2279,0.261653,285,0.261708,285,0.261708
SIRA,2119,0.194618,1695,0.194604,212,0.194674,212,0.194674
SEKER,1613,0.148145,1290,0.148106,162,0.148760,161,0.147842
HOROZ,1522,0.139787,1218,0.139839,152,0.139578,152,0.139578
CALI,1306,0.119949,1045,0.119977,130,0.119376,131,0.120294
BARBUNYA,1063,0.097630,850,0.097589,106,0.097337,107,0.098255
BOMBAY,416,0.038207,333,0.038232,42,0.038567,41,0.037649
